# 20 Preguntas de Negocio

In [16]:
# PostgreSQL connection
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os

load_dotenv()

def connect_to_db():
    try:
        connection = psycopg2.connect(
            dbname=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT")
        )
        print("Connection to PostgreSQL database successful")
        return connection
    except Exception as e:
        print(f"Error connecting to PostgreSQL database: {e}")
        return None
    
days_list = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

months_list = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

In [ ]:
# Count rows per year in the taxi_trips table
def count_rows_per_year(connection):
    try:
        cursor = connection.cursor()
        query = sql.SQL("""
            SELECT LEFT(source_month, 4) AS year, COUNT(*) AS trip_count
            FROM raw.raw_taxi_trips
            GROUP BY year
            ORDER BY year;
        """)
        cursor.execute(query)
        results = cursor.fetchall()
        print("Number of rows per year in the raw.raw_taxi_trips table:")
        for row in results:
            print(f"Year: {int(row[0])}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error counting rows per year in the raw.raw_taxi_trips table: {e}")
    finally:
        cursor.close()

if __name__ == "__main__":
    connection = connect_to_db()
    if connection:
        count_rows_per_year(connection)
        connection.close()

Connection to PostgreSQL database successful
Number of rows per year in the raw.raw_taxi_trips table:
Year: 2022, Trip Count: 40496500
Year: 2023, Trip Count: 39097286
Year: 2024, Trip Count: 41829938
Year: 2025, Trip Count: 44960735


In [ ]:
# Check missing months per year in the taxi_trips table
def check_missing_months_per_year(connection):
    try:
        cursor = connection.cursor()
        query = sql.SQL("""
            WITH months AS (
                SELECT DISTINCT LEFT(source_month, 4) AS year, RIGHT(source_month, 2) AS month
                FROM raw.raw_taxi_trips
            ),
            all_months AS (
                SELECT year, month
                FROM (SELECT DISTINCT LEFT(source_month, 4) AS year FROM raw.raw_taxi_trips) years,
                     (SELECT LPAD(generate_series(1, 12)::text, 2, '0') AS month) months
            )
            SELECT am.year, am.month
            FROM all_months am
            LEFT JOIN months m ON am.year = m.year AND am.month = m.month
            WHERE m.month IS NULL
            ORDER BY am.year, am.month;
        """)
        cursor.execute(query)
        results = cursor.fetchall()
        print("Missing months per year in the raw.raw_taxi_trips table:")
        for row in results:
            print(f"Year: {int(row[0])}, Missing Month: {row[1]}")
    except Exception as e:
        print(f"Error checking missing months per year in the raw.raw_taxi_trips table: {e}")
    finally:
        cursor.close()

if __name__ == "__main__":
    connection = connect_to_db()
    if connection:
        check_missing_months_per_year(connection)
        connection.close()

Connection to PostgreSQL database successful
Missing months per year in the bronze.stg_taxi_trips table:
Year: 2025, Missing Month: 12


### 1. Viajes por mes en 2024:

In [5]:
query_1 = """
SELECT d.month, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.month
ORDER BY d.month;
"""

connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_1)
        results = cursor.fetchall()
        print("Number of rows per month of 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {int(row[0])}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Number of rows per month of 2024 in the analytics_gold.fct_trips table:
Month: 1, Trip Count: 2985411
Month: 2, Trip Count: 3024821
Month: 3, Trip Count: 3595427
Month: 4, Trip Count: 3526712
Month: 5, Trip Count: 3735534
Month: 6, Trip Count: 3544893
Month: 7, Trip Count: 3078104
Month: 8, Trip Count: 2977751
Month: 9, Trip Count: 3631073
Month: 10, Trip Count: 3828252
Month: 11, Trip Count: 3636752
Month: 12, Trip Count: 3651549


### 2. Viajes por *service_type* y mes

In [18]:
query_2 = """
SELECT d.year, d.month, s.service_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s ON f.service_type_key = s.service_type_key
GROUP BY d.year, d.month, s.service_name
ORDER BY d.year, d.month, s.service_name;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_2)
        results = cursor.fetchall()
        print("Number of trips per year, month, and service type in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Year: {int(row[0])}, Month: {months_list[int(row[1] - 1)]}, Service Type: {row[2]}, Trip Count: {row[3]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Number of trips per year, month, and service type in the analytics_gold.fct_trips table:
Year: 2022, Month: January, Service Type: green, Trip Count: 124560
Year: 2022, Month: January, Service Type: yellow, Trip Count: 4899184
Year: 2022, Month: February, Service Type: green, Trip Count: 138352
Year: 2022, Month: February, Service Type: yellow, Trip Count: 5924378
Year: 2022, Month: March, Service Type: green, Trip Count: 156606
Year: 2022, Month: March, Service Type: yellow, Trip Count: 7211972
Year: 2022, Month: April, Service Type: green, Trip Count: 151842
Year: 2022, Month: April, Service Type: yellow, Trip Count: 7157276
Year: 2022, Month: May, Service Type: green, Trip Count: 153408
Year: 2022, Month: May, Service Type: yellow, Trip Count: 7132350
Year: 2022, Month: June, Service Type: green, Trip Count: 147048
Year: 2022, Month: June, Service Type: yellow, Trip Count: 7067390
Year: 2022, Month: July, Service Type: green, Trip Count: 

### 3. Top 10 zonas de pickup (total 2024)

In [9]:
query_3 = """
SELECT z.zone_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone_name
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_3)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the most trips in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the most trips in 2024 in the analytics_gold.fct_trips table:
Zone Name: JFK Airport, Trip Count: 1910212
Zone Name: Upper East Side South, Trip Count: 1892766
Zone Name: Midtown Center, Trip Count: 1886869
Zone Name: Upper East Side North, Trip Count: 1718364
Zone Name: Midtown East, Trip Count: 1399886
Zone Name: Times Sq/Theatre District, Trip Count: 1362163
Zone Name: Penn Station/Madison Sq West, Trip Count: 1340568
Zone Name: Lincoln Square East, Trip Count: 1299317
Zone Name: LaGuardia Airport, Trip Count: 1274993
Zone Name: Murray Hill, Trip Count: 1160460


### 4. Top 10 zonas de dropoff (total 2024)

In [10]:
query_4 = """
SELECT z.zone_name, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.do_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY z.zone_name
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_4)
        results = cursor.fetchall()
        print("Top 10 dropoff zones with the most trips in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 dropoff zones with the most trips in 2024 in the analytics_gold.fct_trips table:
Zone Name: Upper East Side North, Trip Count: 1807540
Zone Name: Upper East Side South, Trip Count: 1714779
Zone Name: Midtown Center, Trip Count: 1521463
Zone Name: Times Sq/Theatre District, Trip Count: 1286396
Zone Name: Murray Hill, Trip Count: 1199473
Zone Name: Midtown East, Trip Count: 1171121
Zone Name: Lincoln Square East, Trip Count: 1137175
Zone Name: Upper West Side South, Trip Count: 1135542
Zone Name: East Chelsea, Trip Count: 1063496
Zone Name: Lenox Hill West, Trip Count: 1058176


### 5. Top 5 *boroughs* por mes (pickup)

In [19]:
query_5 = """
WITH borough_counts AS (
    SELECT d.month, z.borough, COUNT(*) AS total_viajes,
           ROW_NUMBER() OVER(PARTITION BY d.month ORDER BY COUNT(*) DESC) as rank
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
    JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
    GROUP BY d.month, z.borough
)
SELECT month, borough, total_viajes
FROM borough_counts
WHERE rank <= 5
ORDER BY month, total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_5)
        results = cursor.fetchall()
        print("Top 5 boroughs with the most trips per month in 2024 in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Borough: {row[1]}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 5 boroughs with the most trips per month in 2024 in the analytics_gold.fct_trips table:
Month: January, Borough: Manhattan, Trip Count: 10707270
Month: January, Borough: Queens, Trip Count: 1075823
Month: January, Borough: Brooklyn, Trip Count: 155928
Month: January, Borough: Unknown, Trip Count: 81075
Month: January, Borough: Bronx, Trip Count: 34724
Month: February, Borough: Manhattan, Trip Count: 11221813
Month: February, Borough: Queens, Trip Count: 1026254
Month: February, Borough: Brooklyn, Trip Count: 195731
Month: February, Borough: Unknown, Trip Count: 78834
Month: February, Borough: Bronx, Trip Count: 41743
Month: March, Borough: Manhattan, Trip Count: 13073387
Month: March, Borough: Queens, Trip Count: 1343960
Month: March, Borough: Brooklyn, Trip Count: 269615
Month: March, Borough: Unknown, Trip Count: 86991
Month: March, Borough: Bronx, Trip Count: 61014
Month: April, Borough: Manhattan, Trip Count: 12708210
Month: April, B

### 6. Horas pico (top 5) por cada día de la semana

In [20]:
query_6 = """
WITH hourly_counts AS (
    SELECT d.day_of_week, EXTRACT(HOUR FROM f.pickup_ts) AS hora, COUNT(*) AS total_viajes,
           ROW_NUMBER() OVER(PARTITION BY d.day_of_week ORDER BY COUNT(*) DESC) as rank
    FROM analytics_gold.fct_trips f
    JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
    GROUP BY d.day_of_week, hora
)
SELECT day_of_week, hora, total_viajes
FROM hourly_counts
WHERE rank <= 5
ORDER BY day_of_week, total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_6)
        results = cursor.fetchall()
        print("Top 5 hours with the most trips per day of the week in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Day of Week: {days_list[int(row[0]) - 1]}, Hour: {int(row[1])}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 5 hours with the most trips per day of the week in the analytics_gold.fct_trips table:
Day of Week: Monday, Hour: 18, Trip Count: 1502911
Day of Week: Monday, Hour: 17, Trip Count: 1474289
Day of Week: Monday, Hour: 15, Trip Count: 1349761
Day of Week: Monday, Hour: 16, Trip Count: 1340784
Day of Week: Monday, Hour: 14, Trip Count: 1291775
Day of Week: Tuesday, Hour: 18, Trip Count: 1778223
Day of Week: Tuesday, Hour: 17, Trip Count: 1667511
Day of Week: Tuesday, Hour: 19, Trip Count: 1506045
Day of Week: Tuesday, Hour: 21, Trip Count: 1471169
Day of Week: Tuesday, Hour: 16, Trip Count: 1469483
Day of Week: Wednesday, Hour: 18, Trip Count: 1852493
Day of Week: Wednesday, Hour: 17, Trip Count: 1731430
Day of Week: Wednesday, Hour: 19, Trip Count: 1629727
Day of Week: Wednesday, Hour: 21, Trip Count: 1557395
Day of Week: Wednesday, Hour: 16, Trip Count: 1514376
Day of Week: Thursday, Hour: 18, Trip Count: 1899799
Day of Week: Thursday, Hou

### 7. Distribución de viajes por día de la semana (ranking)

In [21]:
query_7 = """
SELECT d.day_of_week, COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.day_of_week
ORDER BY total_viajes DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_7)
        results = cursor.fetchall()
        print("Total trips per day of the week in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Day of Week: {days_list[int(row[0]) - 1]}, Trip Count: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total trips per day of the week in the analytics_gold.fct_trips table:
Day of Week: Thursday, Trip Count: 25567049
Day of Week: Wednesday, Trip Count: 24674116
Day of Week: Friday, Trip Count: 24620965
Day of Week: Saturday, Trip Count: 24597764
Day of Week: Tuesday, Trip Count: 23422015
Day of Week: Sunday, Trip Count: 20873448
Day of Week: Monday, Trip Count: 20429426


### 8. Ingreso total por mes

In [25]:
query_8 = """
SELECT d.month, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_8)
        results = cursor.fetchall()
        print("Total income per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Total Income: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total income per month in the analytics_gold.fct_trips table:
Month: January, Total Income: 306813207.14
Month: February, Total Income: 314218156.24
Month: March, Total Income: 385060198.68
Month: April, Total Income: 381312241.88
Month: May, Total Income: 421143328.69
Month: June, Total Income: 402521154.70
Month: July, Total Income: 352388085.17
Month: August, Total Income: 338092739.22
Month: September, Total Income: 389677515.39
Month: October, Total Income: 424087141.96
Month: November, Total Income: 385454309.34
Month: December, Total Income: 291732404.54


### 9. Ingreso total por *service_type* y mes

In [26]:
query_9 = """
SELECT d.month, s.service_name, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
JOIN analytics_gold.dim_service_type s ON f.service_type_key = s.service_type_key
GROUP BY d.month, s.service_name
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_9)
        results = cursor.fetchall()
        print("Total income per month and service type in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Service Type: {row[1]}, Total Income: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total income per month and service type in the analytics_gold.fct_trips table:
Month: January, Service Type: green, Total Income: 9891933.72
Month: January, Service Type: yellow, Total Income: 603734480.56
Month: February, Service Type: green, Total Income: 9867924.18
Month: February, Service Type: yellow, Total Income: 618568388.30
Month: March, Service Type: green, Total Income: 11190832.40
Month: March, Service Type: yellow, Total Income: 758929564.96
Month: April, Service Type: green, Total Income: 11083985.38
Month: April, Service Type: yellow, Total Income: 751540498.38
Month: May, Service Type: green, Total Income: 12132929.40
Month: May, Service Type: yellow, Total Income: 830153727.98
Month: June, Service Type: green, Total Income: 11408905.22
Month: June, Service Type: yellow, Total Income: 793633404.18
Month: July, Service Type: green, Total Income: 10562662.56
Month: July, Service Type: yellow, Total Income: 694213507.78
Month: A

### 10. tip% promedio por mes

In [28]:
query_10 = """
SELECT d.month, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_10)
        results = cursor.fetchall()
        print("Average tip percentage per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Tip Percentage: {row[1]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average tip percentage per month in the analytics_gold.fct_trips table:
Month: January, Average Tip Percentage: 21.50%
Month: February, Average Tip Percentage: 20.46%
Month: March, Average Tip Percentage: 20.15%
Month: April, Average Tip Percentage: 20.95%
Month: May, Average Tip Percentage: 20.11%
Month: June, Average Tip Percentage: 20.48%
Month: July, Average Tip Percentage: 20.01%
Month: August, Average Tip Percentage: 19.90%
Month: September, Average Tip Percentage: 19.49%
Month: October, Average Tip Percentage: 20.45%
Month: November, Average Tip Percentage: 20.31%
Month: December, Average Tip Percentage: 21.38%


### 11. tip% por *borough* y mes

In [30]:
query_11 = """
SELECT d.month, z.borough, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough, d.month
ORDER BY d.month, avg_tip_pct DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_11)
        results = cursor.fetchall()
        print("Average tip percentage per month and borough in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Borough: {row[1]}, Average Tip Percentage: {row[2]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average tip percentage per month and borough in the analytics_gold.fct_trips table:
Month: January, Borough: EWR, Average Tip Percentage: 47.17%
Month: January, Borough: Manhattan, Average Tip Percentage: 21.21%
Month: January, Borough: Unknown, Average Tip Percentage: 20.16%
Month: January, Borough: Queens, Average Tip Percentage: 19.40%
Month: January, Borough: Staten Island, Average Tip Percentage: 15.42%
Month: January, Borough: Brooklyn, Average Tip Percentage: 9.12%
Month: January, Borough: Bronx, Average Tip Percentage: 4.07%
Month: February, Borough: EWR, Average Tip Percentage: 415.34%
Month: February, Borough: Unknown, Average Tip Percentage: 21.04%
Month: February, Borough: Manhattan, Average Tip Percentage: 20.92%
Month: February, Borough: Queens, Average Tip Percentage: 16.02%
Month: February, Borough: Brooklyn, Average Tip Percentage: 9.29%
Month: February, Borough: Staten Island, Average Tip Percentage: 7.72%
Month: February, 

### 12. Top 10 zonas por ingreso total (pickup)

In [31]:
query_12 = """
SELECT z.zone_name, SUM(total_amount) AS ingreso_total
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
ORDER BY ingreso_total DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_12)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the highest total income in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Total Income: {row[1]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the highest total income in the analytics_gold.fct_trips table:
Zone Name: JFK Airport, Total Income: 575931894.53
Zone Name: LaGuardia Airport, Total Income: 304471456.37
Zone Name: Midtown Center, Total Income: 169643637.96
Zone Name: Upper East Side South, Total Income: 145404022.56
Zone Name: Times Sq/Theatre District, Total Income: 140235336.76
Zone Name: Upper East Side North, Total Income: 133153140.21
Zone Name: Penn Station/Madison Sq West, Total Income: 126387228.20
Zone Name: Midtown East, Total Income: 124328670.91
Zone Name: Lincoln Square East, Total Income: 106566263.40
Zone Name: Midtown North, Total Income: 106061840.22


### 13. Top 10 zonas por tip% (pickup) con mínimo N viajes (N documentado)

In [33]:
N = 10000  # Definimos N como 10,000 viajes para significancia estadística

query_13 = f"""
SELECT z.zone_name, AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct, COUNT(*) as total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
HAVING COUNT(*) >= {N}
ORDER BY avg_tip_pct DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_13)
        results = cursor.fetchall()
        print(f"Top 10 pickup zones (with at least {N} trips)with the highest average tip percentage in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, Average Tip Percentage: {row[1]:.2f}%, Total Trips: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones (with at least 10000 trips)with the highest average tip percentage in the analytics_gold.fct_trips table:
Zone Name: Outside of NYC, Average Tip Percentage: 565.31%, Total Trips: 143788
Zone Name: Newark Airport, Average Tip Percentage: 388.92%, Total Trips: 26001
Zone Name: Baisley Park, Average Tip Percentage: 82.47%, Total Trips: 63211
Zone Name: Spuyten Duyvil/Kingsbridge, Average Tip Percentage: 67.87%, Total Trips: 14943
Zone Name: East New York, Average Tip Percentage: 60.70%, Total Trips: 83737
Zone Name: Norwood, Average Tip Percentage: 53.46%, Total Trips: 13545
Zone Name: Red Hook, Average Tip Percentage: 35.55%, Total Trips: 41648
Zone Name: Springfield Gardens South, Average Tip Percentage: 34.95%, Total Trips: 26539
Zone Name: East Concourse/Concourse Village, Average Tip Percentage: 31.56%, Total Trips: 32484
Zone Name: Williamsbridge/Olinville, Average Tip Percentage: 29.96%, Total Trips: 21751


### 14. Comparación *cash vs card*: viajes, ingreso total, tip%

In [35]:
query_14 = """
SELECT p.payment_type_name AS payment_method, 
       COUNT(*) AS total_viajes, 
       SUM(total_amount) AS ingreso_total,
       AVG(tip_amount / NULLIF(fare_amount, 0)) * 100 AS avg_tip_pct
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_payment_type p ON f.payment_type_key = p.payment_type_key
WHERE p.payment_type_key IN (1, 2)  -- 1 = Cash, 2 = Card
GROUP BY p.payment_type_name;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_14)
        results = cursor.fetchall()
        print("Total trips, total income, and average tip percentage by payment method (cash v. card) in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Payment Method: {row[0]}, Total Trips: {row[1]}, Total Income: {row[2]}, Average Tip Percentage: {row[3]:.2f}%")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Total trips, total income, and average tip percentage by payment method (cash v. card) in the analytics_gold.fct_trips table:
Payment Method: Cash, Total Trips: 24291078, Total Income: 554895557.53, Average Tip Percentage: 0.00%
Payment Method: Credit card, Total Trips: 120611123, Total Income: 3378589885.38, Average Tip Percentage: 27.14%


### 15. Duración promedio (min) por mes

In [37]:
query_15 = """
SELECT d.month, AVG(EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS avg_duration_min
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_15)
        results = cursor.fetchall()
        print("Average trip duration in minutes per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Duration: {row[1]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip duration in minutes per month in the analytics_gold.fct_trips table:
Month: January, Average Duration: 15.27 minutes
Month: February, Average Duration: 15.85 minutes
Month: March, Average Duration: 16.53 minutes
Month: April, Average Duration: 17.12 minutes
Month: May, Average Duration: 18.12 minutes
Month: June, Average Duration: 17.76 minutes
Month: July, Average Duration: 17.14 minutes
Month: August, Average Duration: 17.28 minutes
Month: September, Average Duration: 18.74 minutes
Month: October, Average Duration: 18.39 minutes
Month: November, Average Duration: 18.19 minutes
Month: December, Average Duration: 18.48 minutes


### 16. Distancia promedio por mes

In [38]:
query_16 = """
SELECT d.month, AVG(trip_distance) AS avg_distance
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_date d ON f.pickup_date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_16)
        results = cursor.fetchall()
        print("Average trip distance per month in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Month: {months_list[int(row[0]) - 1]}, Average Distance: {row[1]:.2f} miles")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip distance per month in the analytics_gold.fct_trips table:
Month: January, Average Distance: 5.30 miles
Month: February, Average Distance: 5.62 miles
Month: March, Average Distance: 5.71 miles
Month: April, Average Distance: 6.14 miles
Month: May, Average Distance: 6.85 miles
Month: June, Average Distance: 6.51 miles
Month: July, Average Distance: 6.32 miles
Month: August, Average Distance: 6.17 miles
Month: September, Average Distance: 6.75 miles
Month: October, Average Distance: 6.06 miles
Month: November, Average Distance: 6.02 miles
Month: December, Average Distance: 4.94 miles


### 17. Velocidad promedio (mph) por *borough* y franja horaria

In [41]:
query_17 = """
SELECT z.borough, 
       CASE 
           WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 6 AND 9 THEN 'Morning Rush'
           WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 16 AND 19 THEN 'Evening Rush'
           ELSE 'Other'
       END AS time_slot,
       AVG(trip_distance / (NULLIF(EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/3600, 0))) AS avg_speed_mph
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough, time_slot
ORDER BY z.borough, avg_speed_mph DESC;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_17)
        results = cursor.fetchall()
        print("Average trip speed in mph by borough and time slot in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Borough: {row[0]}, Time Slot: {row[1]}, Average Speed: {row[2]:.2f} mph")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Average trip speed in mph by borough and time slot in the analytics_gold.fct_trips table:
Borough: Bronx, Time Slot: Morning Rush, Average Speed: 218.17 mph
Borough: Bronx, Time Slot: Other, Average Speed: 175.34 mph
Borough: Bronx, Time Slot: Evening Rush, Average Speed: 100.38 mph
Borough: Brooklyn, Time Slot: Morning Rush, Average Speed: 167.38 mph
Borough: Brooklyn, Time Slot: Other, Average Speed: 82.77 mph
Borough: Brooklyn, Time Slot: Evening Rush, Average Speed: 80.84 mph
Borough: EWR, Time Slot: Morning Rush, Average Speed: 101.24 mph
Borough: EWR, Time Slot: Evening Rush, Average Speed: 65.97 mph
Borough: EWR, Time Slot: Other, Average Speed: 61.63 mph
Borough: Manhattan, Time Slot: Morning Rush, Average Speed: 31.20 mph
Borough: Manhattan, Time Slot: Other, Average Speed: 18.69 mph
Borough: Manhattan, Time Slot: Evening Rush, Average Speed: 15.25 mph
Borough: Queens, Time Slot: Morning Rush, Average Speed: 58.32 mph
Borough: Queen

### 18. Percentiles 50 y 90 de duración por borough

In [42]:
query_18 = """
SELECT z.borough, 
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p90_duration
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE z.borough IS NOT NULL
GROUP BY z.borough;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_18)
        results = cursor.fetchall()
        print("50th and 90th percentile of trip duration in minutes by borough in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Borough: {row[0]}, P50 Duration: {row[1]:.2f} minutes, P90 Duration: {row[2]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
50th and 90th percentile of trip duration in minutes by borough in the analytics_gold.fct_trips table:
Borough: Bronx, P50 Duration: 23.13 minutes, P90 Duration: 61.47 minutes
Borough: Brooklyn, P50 Duration: 21.50 minutes, P90 Duration: 48.53 minutes
Borough: EWR, P50 Duration: 0.15 minutes, P90 Duration: 1.32 minutes
Borough: Manhattan, P50 Duration: 11.88 minutes, P90 Duration: 26.30 minutes
Borough: Queens, P50 Duration: 31.88 minutes, P90 Duration: 60.75 minutes
Borough: Staten Island, P50 Duration: 29.03 minutes, P90 Duration: 72.59 minutes
Borough: Unknown, P50 Duration: 14.00 minutes, P90 Duration: 38.18 minutes


### 19. Top 10 zonas (pickup) por percentil 90 de duración

In [43]:
query_19 = """
SELECT z.zone_name, 
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY EXTRACT(EPOCH FROM (dropoff_ts - pickup_ts))/60) AS p90_duration
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY z.zone_name
ORDER BY p90_duration DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_19)
        results = cursor.fetchall()
        print("Top 10 pickup zones with the longest 90th percentile trip duration in minutes in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Zone Name: {row[0]}, P90 Duration: {row[1]:.2f} minutes")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup zones with the longest 90th percentile trip duration in minutes in the analytics_gold.fct_trips table:
Zone Name: Arden Heights, P90 Duration: 124.04 minutes
Zone Name: Freshkills Park, P90 Duration: 102.09 minutes
Zone Name: Far Rockaway, P90 Duration: 99.98 minutes
Zone Name: Hammels/Arverne, P90 Duration: 97.36 minutes
Zone Name: Coney Island, P90 Duration: 92.65 minutes
Zone Name: Eltingville/Annadale/Prince's Bay, P90 Duration: 90.57 minutes
Zone Name: Rockaway Park, P90 Duration: 88.18 minutes
Zone Name: Brighton Beach, P90 Duration: 84.07 minutes
Zone Name: Gravesend, P90 Duration: 83.14 minutes
Zone Name: Cambria Heights, P90 Duration: 80.80 minutes


### 20. Top 10 rutas *borough to borough* por número de viajes

In [46]:
query_20 = """
SELECT z_pu.borough AS pickup_borough, 
       z_do.borough AS dropoff_borough, 
       COUNT(*) AS total_viajes
FROM analytics_gold.fct_trips f
JOIN analytics_gold.dim_zone z_pu ON f.pu_zone_key = z_pu.zone_key
JOIN analytics_gold.dim_zone z_do ON f.do_zone_key = z_do.zone_key
WHERE z_pu.borough IS NOT NULL AND z_do.borough IS NOT NULL
GROUP BY pickup_borough, dropoff_borough
ORDER BY total_viajes DESC
LIMIT 10;
"""
connection = connect_to_db()
if connection:
    try:
        cursor = connection.cursor()
        cursor.execute(query_20)
        results = cursor.fetchall()
        print("Top 10 pickup and dropoff borough pairs with the most trips in the analytics_gold.fct_trips table:")
        for row in results:
            print(f"Pickup Borough: {row[0]}, Dropoff Borough: {row[1]}, Trip Count: {row[2]}")
    except Exception as e:
        print(f"Error executing query: {e}")
    finally:
        cursor.close()
        connection.close()

Connection to PostgreSQL database successful
Top 10 pickup and dropoff borough pairs with the most trips in the analytics_gold.fct_trips table:
Pickup Borough: Manhattan, Dropoff Borough: Manhattan, Trip Count: 133948087
Pickup Borough: Queens, Dropoff Borough: Manhattan, Trip Count: 8271378
Pickup Borough: Manhattan, Dropoff Borough: Queens, Trip Count: 4706915
Pickup Borough: Queens, Dropoff Borough: Queens, Trip Count: 4283222
Pickup Borough: Manhattan, Dropoff Borough: Brooklyn, Trip Count: 3368958
Pickup Borough: Queens, Dropoff Borough: Brooklyn, Trip Count: 2265575
Pickup Borough: Brooklyn, Dropoff Borough: Brooklyn, Trip Count: 1754350
Pickup Borough: Brooklyn, Dropoff Borough: Manhattan, Trip Count: 971410
Pickup Borough: Manhattan, Dropoff Borough: Bronx, Trip Count: 638222
Pickup Borough: Unknown, Dropoff Borough: Unknown, Trip Count: 590788
